# Week 1 Baseline and project Setup

Pilot study: Deployment-Realistic Compression for Low-Resource NMT.

**What this notebook does:** [One sentence]
**Runtime:** [X hours on T4×2 / CPU-only / etc.]
**Outputs:** [Which CSV files it produces, in which folder]
**Dependencies:** See `requirements.txt` in repo root.

For context, see the project [README](../README.md) and [technical report](../writeup/technical_report.md).

In [31]:
# Pin exact versions for reproducibility
!pip install -q \
    transformers==4.44.2 \
    bitsandbytes==0.43.3 \
    accelerate==0.34.2 \
    datasets==3.0.1 \
    evaluate==0.4.3 \
    sacrebleu==2.4.3 \
    sentencepiece==0.2.0

# Verify GPU
import torch
assert torch.cuda.is_available(), "No GPU — switch runtime to T4"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)
GPU: Tesla T4
VRAM: 15.6 GB


In [32]:
import os
from pathlib import Path
from datetime import datetime

BASE = Path('/kaggle/working/pilot')
for sub in ['notebooks', 'results', 'checkpoints', 'logs']:
    (BASE / sub).mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now().strftime('%Y-%m-%d_%H%M')

In [33]:

from kaggle_secrets import UserSecretsClient
from datasets import load_dataset

# Authenticate using your Kaggle secret
hf_token = UserSecretsClient().get_secret("HF_TOKEN")

# FLORES-200 devtest: 1,012 parallel sentences per language
flores_hau = load_dataset(
    "facebook/flores",
    "hau_Latn",
    split="devtest",
    token=hf_token,
)

flores_swh = load_dataset(
    "facebook/flores",
    "swh_Latn",
    split="devtest",
    token=hf_token,
)

flores_eng = load_dataset(
    "facebook/flores",
    "eng_Latn",
    split="devtest",
    token=hf_token,
)

print(f"Hausa: {len(flores_hau)} sentences")
print(f"Swahili: {len(flores_swh)} sentences")
print(f"English: {len(flores_eng)} sentences")

# Inspect aligned sentences
print("\nSample index 0:")
print(f"  HAU: {flores_hau[0]['sentence'][:100]}")
print(f"  SWH: {flores_swh[0]['sentence'][:100]}")
print(f"  ENG: {flores_eng[0]['sentence'][:100]}")


Hausa: 1012 sentences
Swahili: 1012 sentences
English: 1012 sentences

Sample index 0:
  HAU: “Yanzu muna bera mai watanni 4 wadanda ba su da ciwon sukari ba wadanda ke da suke fama da ciwon suk
  SWH: "Sasa tuna panya wenye umri wa miezi minne ambao hawana kisukari ambao awali walikuwa na kisukari", 
  ENG: "We now have 4-month-old mice that are non-diabetic that used to be diabetic," he added.


In [34]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    BitsAndBytesConfig,
)
import torch
import gc

MODELS = {
    "qwen-1.5b": {
        "name": "Qwen/Qwen2.5-1.5B-Instruct",
        "arch": "causal",
    },
    "aya-8b": {
        "name": "CohereForAI/aya-23-8B",
        "arch": "causal",
    },
    "nllb-1.3b": {
        "name": "facebook/nllb-200-distilled-1.3B",
        "arch": "seq2seq",
    },
}

# NLLB uses FLORES-style language codes
NLLB_LANG_CODES = {
    "hau": "hau_Latn",
    "swh": "swh_Latn",
    "eng": "eng_Latn",
}

def load_model(model_key, precision="fp16"):
    """Load specified model in specified precision. Handles causal and seq2seq."""
    cfg = MODELS[model_key]
    model_name = cfg["name"]
    arch = cfg["arch"]

    tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Pick the right AutoModel class
    model_cls = AutoModelForCausalLM if arch == "causal" else AutoModelForSeq2SeqLM

    if precision == "fp16":
        model = model_cls.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="auto",
        )
    elif precision == "int8":
        bnb_cfg = BitsAndBytesConfig(load_in_8bit=True)
        model = model_cls.from_pretrained(
            model_name,
            quantization_config=bnb_cfg,
            device_map="auto",
        )
    elif precision == "int4":
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
        model = model_cls.from_pretrained(
            model_name,
            quantization_config=bnb_cfg,
            device_map="auto",
        )
    else:
        raise ValueError(f"Unknown precision: {precision}")

    model.eval()
    total_mem = sum(
        torch.cuda.memory_allocated(i) for i in range(torch.cuda.device_count())
    ) / 1e9
    print(f"  Loaded {model_name} ({arch}, {precision}), GPU memory: {total_mem:.2f} GB")
    return model, tokenizer

def free_model(model):
    """Clear GPU memory before loading next model."""
    del model
    gc.collect()
    torch.cuda.empty_cache()

In [35]:
# Prompt templates for causal LMs (Qwen uses explicit, Aya uses minimal)
TRANSLATION_PROMPTS = {
    ("hau", "eng"): "Translate the following Hausa sentence to English. Output only the translation, nothing else.\n\nHausa: {src}\n\nEnglish:",
    ("eng", "hau"): "Translate the following English sentence to Hausa. Output only the translation, nothing else.\n\nEnglish: {src}\n\nHausa:",
    ("swh", "eng"): "Translate the following Swahili sentence to English. Output only the translation, nothing else.\n\nSwahili: {src}\n\nEnglish:",
    ("eng", "swh"): "Translate the following English sentence to Swahili. Output only the translation, nothing else.\n\nEnglish: {src}\n\nSwahili:",
}

AYA_TRANSLATION_PROMPTS = {
    ("hau", "eng"): "Translate from Hausa to English: {src}",
    ("eng", "hau"): "Translate from English to Hausa: {src}",
    ("swh", "eng"): "Translate from Swahili to English: {src}",
    ("eng", "swh"): "Translate from English to Swahili: {src}",
}

def get_causal_prompts(model_key):
    return AYA_TRANSLATION_PROMPTS if model_key == "aya-8b" else TRANSLATION_PROMPTS


def translate_causal_batch(model, tokenizer, sentences, src_lang, tgt_lang, model_key,
                           batch_size=8, max_new_tokens=256):
    """Translate using a causal LM (Qwen, Aya) with chat template."""
    prompts_dict = get_causal_prompts(model_key)
    prompt_tpl = prompts_dict[(src_lang, tgt_lang)]
    translations = []

    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        messages_batch = [
            [{"role": "user", "content": prompt_tpl.format(src=s)}] for s in batch
        ]
        prompts = [
            tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
            for m in messages_batch
        ]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True,
                           truncation=True, max_length=512).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=1,
                pad_token_id=tokenizer.pad_token_id,
            )

        new_tokens = outputs[:, inputs['input_ids'].shape[1]:]
        decoded = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
        translations.extend([d.strip() for d in decoded])

        if i % (batch_size * 10) == 0:
            print(f"  Translated {i+len(batch)}/{len(sentences)}")

    return translations


def translate_nllb_batch(model, tokenizer, sentences, src_lang, tgt_lang,
                         batch_size=16, max_new_tokens=256):
    """Translate using NLLB-200 (seq2seq, needs forced BOS token for target language)."""
    src_code = NLLB_LANG_CODES[src_lang]
    tgt_code = NLLB_LANG_CODES[tgt_lang]

    # NLLB conventions: set src_lang on the tokenizer, force target BOS at generation
    tokenizer.src_lang = src_code
    forced_bos = tokenizer.convert_tokens_to_ids(tgt_code)

    translations = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        inputs = tokenizer(
            batch, return_tensors="pt", padding=True, truncation=True, max_length=512
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                forced_bos_token_id=forced_bos,
                max_new_tokens=max_new_tokens,
                num_beams=1,
                do_sample=False,
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        translations.extend([d.strip() for d in decoded])

        if i % (batch_size * 10) == 0:
            print(f"  Translated {i+len(batch)}/{len(sentences)}")

    return translations


def translate_batch(model, tokenizer, sentences, src_lang, tgt_lang, model_key,
                    batch_size=8, max_new_tokens=256):
    """Dispatch to the right translation function based on model architecture."""
    arch = MODELS[model_key]["arch"]
    if arch == "seq2seq":
        # NLLB tolerates larger batches since it's smaller and seq2seq
        bs = max(batch_size, 16)
        return translate_nllb_batch(model, tokenizer, sentences, src_lang, tgt_lang,
                                    batch_size=bs, max_new_tokens=max_new_tokens)
    else:
        return translate_causal_batch(model, tokenizer, sentences, src_lang, tgt_lang,
                                      model_key, batch_size=batch_size,
                                      max_new_tokens=max_new_tokens)

In [36]:
!pip install sacrebleu

In [37]:
import sacrebleu
import time
import pandas as pd

def evaluate_direction(model, tokenizer, src_data, tgt_data, src_lang, tgt_lang,
                       precision, model_key, n_samples=200, batch_size=8):
    """Run one translation direction and compute metrics. Works for all model architectures."""
    src_sents = [x['sentence'] for x in src_data.select(range(n_samples))]
    tgt_refs = [x['sentence'] for x in tgt_data.select(range(n_samples))]

    print(f"\n→ [{model_key}] {src_lang}→{tgt_lang} @ {precision}, n={n_samples}")
    t0 = time.time()
    hypotheses = translate_batch(model, tokenizer, src_sents, src_lang, tgt_lang,
                                 model_key, batch_size=batch_size)
    elapsed = time.time() - t0

    bleu = sacrebleu.corpus_bleu(hypotheses, [tgt_refs]).score
    chrf = sacrebleu.corpus_chrf(hypotheses, [tgt_refs]).score

    total_mem = sum(
        torch.cuda.max_memory_allocated(i) for i in range(torch.cuda.device_count())
    ) / 1e9

    result = {
        'run_id': RUN_ID,
        'model_key': model_key,
        'model_name': MODELS[model_key]['name'],
        'arch': MODELS[model_key]['arch'],
        'precision': precision,
        'src_lang': src_lang,
        'tgt_lang': tgt_lang,
        'n_samples': n_samples,
        'bleu': round(bleu, 2),
        'chrf': round(chrf, 2),
        'total_seconds': round(elapsed, 1),
        'sec_per_sentence': round(elapsed / n_samples, 3),
        'gpu_mem_gb': round(total_mem, 2),
    }
    print(f"  BLEU: {result['bleu']}, chrF: {result['chrf']}, time: {result['total_seconds']}s")

    hyp_path = BASE / 'results' / f'hyps_{model_key}_{src_lang}-{tgt_lang}_{precision}_{RUN_ID}.txt'
    with open(hyp_path, 'w', encoding='utf-8') as f:
        for s, h, r in zip(src_sents, hypotheses, tgt_refs):
            f.write(f"SRC: {s}\nHYP: {h}\nREF: {r}\n\n")

    return result

In [38]:
%pip install -U "bitsandbytes>=0.46.1" sentencepiece==0.2.0

Note: you may need to restart the kernel to use updated packages.


In [39]:
import torch
import transformers
import bitsandbytes as bnb

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)
print("BitsAndBytes:", bnb.__version__)

PyTorch: 2.11.0+cu128
CUDA available: True
Transformers: 5.16.1
BitsAndBytes: 0.50.2


In [40]:
def free_model(model):
    """Clear GPU memory before loading next model."""
    del model
    gc.collect()
    torch.cuda.empty_cache()

In [41]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

login(token=UserSecretsClient().get_secret("HF_TOKEN"))

In [42]:
# Precision sweep per model — NLLB uses INT8 (works better than INT4 at this scale);
# causal LMs sweep FP16/INT4 as before, with INT8 added for a fuller picture.

MODEL_PRECISION_SWEEP = {
    "qwen-1.5b": ["fp16", "int8", "int4"],
    "aya-8b":    ["fp16", "int8", "int4"],
    "nllb-1.3b": ["fp16", "int8", "int4"],
}

N_SAMPLES = 1012  # Full devtest
all_results = []

for model_key, precisions in MODEL_PRECISION_SWEEP.items():
    for precision in precisions:
        print(f"\n{'='*60}\n{model_key} / {precision}\n{'='*60}")
        torch.cuda.reset_peak_memory_stats()
        try:
            model, tokenizer = load_model(model_key, precision)
        except Exception as e:
            print(f"  ✗ Failed to load: {e}")
            continue

        for src_lang, tgt_lang, src_data, tgt_data in DIRECTIONS:
            try:
                result = evaluate_direction(
                    model, tokenizer, src_data, tgt_data,
                    src_lang, tgt_lang, precision,
                    model_key=model_key, n_samples=N_SAMPLES
                )
                all_results.append(result)
                pd.DataFrame(all_results).to_csv(
                    BASE / 'results' / f'baseline_week2_{RUN_ID}_partial.csv', index=False
                )
            except Exception as e:
                print(f"  ✗ Failed on {src_lang}->{tgt_lang}: {e}")
                continue

        free_model(model)

df = pd.DataFrame(all_results)
df.to_csv(BASE / 'results' / f'baseline_week2_{RUN_ID}.csv', index=False)
print(f"\n✓ Week 2 baseline sweep saved")
df


qwen-1.5b / fp16


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  Loaded Qwen/Qwen2.5-1.5B-Instruct (fp16), total GPU memory: 8.81 GB

→ [qwen-1.5b] hau→eng @ fp16, n=1012
  Translated 8/1012
  Translated 88/1012
  Translated 168/1012
  Translated 248/1012
  Translated 328/1012
  Translated 408/1012
  Translated 488/1012
  Translated 568/1012
  Translated 648/1012
  Translated 728/1012
  Translated 808/1012
  Translated 888/1012
  Translated 968/1012
  BLEU: 2.89, chrF: 23.69, time: 409.1s

→ [qwen-1.5b] eng→hau @ fp16, n=1012
  Translated 8/1012
  Translated 88/1012
  Translated 168/1012
  Translated 248/1012
  Translated 328/1012
  Translated 408/1012
  Translated 488/1012
  Translated 568/1012
  Translated 648/1012
  Translated 728/1012
  Translated 808/1012
  Translated 888/1012
  Translated 968/1012
  BLEU: 0.84, chrF: 11.23, time: 1260.3s

→ [qwen-1.5b] swh→eng @ fp16, n=1012
  Translated 8/1012
  Translated 88/1012
  Translated 168/1012
  Translated 248/1012
  Translated 328/1012
  Translated 408/1012
  Translated 488/1012
  Translated 568/1

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  Loaded Qwen/Qwen2.5-1.5B-Instruct (int4), total GPU memory: 4.26 GB

→ [qwen-1.5b] hau→eng @ int4, n=1012
  Translated 8/1012
  Translated 88/1012
  Translated 168/1012
  Translated 248/1012
  Translated 328/1012
  Translated 408/1012
  Translated 488/1012
  Translated 568/1012
  Translated 648/1012
  Translated 728/1012
  Translated 808/1012
  Translated 888/1012
  Translated 968/1012
  BLEU: 2.69, chrF: 23.09, time: 829.9s

→ [qwen-1.5b] eng→hau @ int4, n=1012
  Translated 8/1012
  Translated 88/1012
  Translated 168/1012
  Translated 248/1012
  Translated 328/1012
  Translated 408/1012
  Translated 488/1012
  Translated 568/1012
  Translated 648/1012
  Translated 728/1012
  Translated 808/1012
  Translated 888/1012
  Translated 968/1012
  BLEU: 1.35, chrF: 18.64, time: 2284.7s

→ [qwen-1.5b] swh→eng @ int4, n=1012
  Translated 8/1012
  Translated 88/1012
  Translated 168/1012
  Translated 248/1012
  Translated 328/1012
  Translated 408/1012
  Translated 488/1012
  Translated 568/1

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

  Loaded CohereForAI/aya-23-8B (fp16), total GPU memory: 17.23 GB

→ [aya-8b] hau→eng @ fp16, n=1012
  Translated 8/1012
  Translated 88/1012
  Translated 168/1012
  Translated 248/1012
  Translated 328/1012
  Translated 408/1012
  Translated 488/1012
  Translated 568/1012
  Translated 648/1012
  Translated 728/1012
  Translated 808/1012
  Translated 888/1012
  Translated 968/1012
  BLEU: 2.7, chrF: 20.19, time: 1361.0s

→ [aya-8b] eng→hau @ fp16, n=1012
  Translated 8/1012
  Translated 88/1012
  Translated 168/1012
  Translated 248/1012
  Translated 328/1012
  Translated 408/1012
  Translated 488/1012
  Translated 568/1012
  Translated 648/1012
  Translated 728/1012
  Translated 808/1012
  Translated 888/1012
  Translated 968/1012
  BLEU: 1.21, chrF: 13.69, time: 2390.7s

→ [aya-8b] swh→eng @ fp16, n=1012
  Translated 8/1012
  Translated 88/1012
  Translated 168/1012
  Translated 248/1012
  Translated 328/1012
  Translated 408/1012
  Translated 488/1012
  Translated 568/1012
  Transla

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

  Loaded CohereForAI/aya-23-8B (int4), total GPU memory: 21.78 GB

→ [aya-8b] hau→eng @ int4, n=1012
  Translated 8/1012
  Translated 88/1012
  Translated 168/1012
  Translated 248/1012
  Translated 328/1012
  Translated 408/1012
  Translated 488/1012
  Translated 568/1012
  Translated 648/1012
  Translated 728/1012
  Translated 808/1012
  Translated 888/1012
  Translated 968/1012
  BLEU: 2.52, chrF: 19.85, time: 1842.8s

→ [aya-8b] eng→hau @ int4, n=1012
  Translated 8/1012
  Translated 88/1012
  Translated 168/1012
  Translated 248/1012
  Translated 328/1012
  Translated 408/1012
  Translated 488/1012
  Translated 568/1012
  Translated 648/1012
  Translated 728/1012
  Translated 808/1012
  Translated 888/1012
  Translated 968/1012
  BLEU: 1.08, chrF: 13.71, time: 3270.9s

→ [aya-8b] swh→eng @ int4, n=1012
  Translated 8/1012
  Translated 88/1012
  Translated 168/1012
  Translated 248/1012
  Translated 328/1012
  Translated 408/1012
  Translated 488/1012
  Translated 568/1012
  Transl

,run_id,model_key,model_name,precision,src_lang,tgt_lang,n_samples,bleu,chrf,total_seconds,sec_per_sentence,gpu_mem_gb
0,2026-10-18_0940,qwen-1.5b,Qwen/Qwen2.5-1.5B-Instruct,fp16,hau,eng,1012,2.89,23.69,409.1,0.404,16.01
1,2026-10-18_0940,qwen-1.5b,Qwen/Qwen2.5-1.5B-Instruct,fp16,eng,hau,1012,0.84,11.23,1260.3,1.245,16.01
2,2026-10-18_0940,qwen-1.5b,Qwen/Qwen2.5-1.5B-Instruct,fp16,swh,eng,1012,3.35,24.04,557.6,0.551,16.02
3,2026-10-18_0940,qwen-1.5b,Qwen/Qwen2.5-1.5B-Instruct,fp16,eng,swh,1012,1.79,23.90,956.0,0.945,16.02
4,2026-10-18_0940,qwen-1.5b,Qwen/Qwen2.5-1.5B-Instruct,int4,hau,eng,1012,2.69,23.09,829.9,0.820,14.05
5,2026-10-18_0940,qwen-1.5b,Qwen/Qwen2.5-1.5B-Instruct,int4,eng,hau,1012,1.35,18.64,2284.7,2.258,14.05
6,2026-10-18_0940,qwen-1.5b,Qwen/Qwen2.5-1.5B-Instruct,int4,swh,eng,1012,2.98,23.99,1005.0,0.993,14.06
7,2026-10-18_0940,qwen-1.5b,Qwen/Qwen2.5-1.5B-Instruct,int4,eng,swh,1012,1.71,23.15,2098.7,2.074,14.06
8,2026-10-18_0940,aya-8b,CohereForAI/aya-23-8B,fp16,hau,eng,1012,2.70,20.19,1361.0,1.345,20.45
9,2026-10-18_0940,aya-8b,CohereForAI/aya-23-8B,fp16,eng,hau,1012,1.21,13.69,2390.7,2.362,20.45


In [48]:
from transformers import AutoTokenizer

print(MODELS.keys())  # see what keys you have
tok = AutoTokenizer.from_pretrained(MODELS[model_key])

test_strings = [
    "ɓaure",      # Hausa, hooked b
    "ɗan",        # Hausa, hooked d
    "ƙasa",       # Hausa, hooked k
    "Habari za asubuhi",  # Swahili greeting
]
for s in test_strings:
    ids = tok.encode(s, add_special_tokens=False)
    pieces = tok.convert_ids_to_tokens(ids)
    roundtrip = tok.decode(ids)
    ok = "OK" if roundtrip == s else f"MISMATCH → {roundtrip!r}"
    print(f"{s!r:30} → {len(ids)} tokens: {pieces}  [{ok}]")

dict_keys(['qwen-1.5b', 'aya-8b'])
'ɓaure'                        → 3 tokens: ['É', 'ĵ', 'aure']  [OK]
'ɗan'                          → 3 tokens: ['É', 'Ĺ', 'an']  [OK]
'ƙasa'                         → 3 tokens: ['Æ', 'Ļ', 'asa']  [OK]
'Habari za asubuhi'            → 6 tokens: ['Hab', 'ari', 'Ġza', 'Ġas', 'ubu', 'hi']  [OK]


In [49]:
import shutil

shutil.make_archive(
    "/kaggle/working/results",
    "zip",
    "/kaggle/working"
)

'/kaggle/working/results.zip'

In [50]:
from IPython.display import FileLink, display

#display(FileLink("/kaggle/working/results.zip"))


/kaggle/working/pilot.zip